# 02. マッピング & カウントトリミング済みFASTQをhg38ゲノムにマッピングし、遺伝子ごとのリードカウント行列を生成します。**カーネル**: RNA-seq (Python) / rnaseq_env  **必要ツール**: STAR, featureCounts (Subread), samtools  **入力**: `trimmed/` 内のトリミング済みFASTQ  **出力**: `mapped/` にBAM, `results/count_matrix.csv`

## 設定

In [ ]:
import os, glob, pandas as pd

# === 設定 ===
TRIMMED_DIR = "trimmed"
MAPPED_DIR = "mapped"
RESULTS_DIR = "results"
THREADS = 8

# ★ ユーザーが編集する箇所 ★
GENOME_DIR = "reference/hg38/star_index"   # STARインデックスのパス
GTF_FILE = "reference/hg38/gencode.v44.primary_assembly.annotation.gtf"  # GTFファイルのパス

for d in [MAPPED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# パスの確認
assert os.path.isdir(GENOME_DIR), f"STARインデックスが見つかりません: {GENOME_DIR}"
assert os.path.isfile(GTF_FILE), f"GTFファイルが見つかりません: {GTF_FILE}"
print("設定完了 - リファレンスファイル確認OK")

## サンプルメタデータの読み込み

In [ ]:
metadata = pd.read_csv("sample_metadata.csv")
print(f"サンプル数: {len(metadata)}")
print(metadata.to_string(index=False))

## Step 1: STAR マッピング

In [ ]:
%%time
for _, row in metadata.iterrows():
    sample_id = row["sample_id"]
    
    # トリミング後のファイル名パターンを検出
    r1_pattern = os.path.join(TRIMMED_DIR, f"*{sample_id}*_val_1.fq.gz")
    r2_pattern = os.path.join(TRIMMED_DIR, f"*{sample_id}*_val_2.fq.gz")
    
    r1_files = glob.glob(r1_pattern)
    r2_files = glob.glob(r2_pattern)
    
    if not r1_files or not r2_files:
        # 元のファイル名ベースで探す
        r1_base = os.path.basename(row["fastq_r1"]).replace(".fastq.gz", "_val_1.fq.gz")
        r2_base = os.path.basename(row["fastq_r2"]).replace(".fastq.gz", "_val_2.fq.gz")
        r1_files = [os.path.join(TRIMMED_DIR, r1_base)]
        r2_files = [os.path.join(TRIMMED_DIR, r2_base)]
    
    r1 = r1_files[0]
    r2 = r2_files[0]
    prefix = os.path.join(MAPPED_DIR, f"{sample_id}_")
    
    cmd = (
        f"STAR --runThreadN {THREADS} "
        f"--genomeDir {GENOME_DIR} "
        f"--readFilesIn {r1} {r2} "
        f"--readFilesCommand zcat "
        f"--outFileNamePrefix {prefix} "
        f"--outSAMtype BAM SortedByCoordinate "
        f"--outSAMattributes NH HI AS NM MD "
        f"--quantMode GeneCounts "
        f"--outBAMsortingThreadN {THREADS}"
    )
    
    print(f"マッピング中: {sample_id}")
    os.system(cmd)
    
    # BAMインデックス作成
    bam = f"{prefix}Aligned.sortedByCoord.out.bam"
    os.system(f"samtools index -@ {THREADS} {bam}")

print("\nSTAR マッピング完了")

## Step 2: マッピング統計

In [ ]:
# STARのログから統計を取得
stats = []
for _, row in metadata.iterrows():
    log_file = os.path.join(MAPPED_DIR, f"{row['sample_id']}_Log.final.out")
    if os.path.exists(log_file):
        info = {"sample_id": row["sample_id"]}
        with open(log_file) as f:
            for line in f:
                if "Uniquely mapped reads %" in line:
                    info["unique_map_rate"] = line.split("|")[1].strip()
                elif "Number of input reads" in line:
                    info["total_reads"] = line.split("|")[1].strip()
        stats.append(info)

stats_df = pd.DataFrame(stats)
print("マッピング統計:")
print(stats_df.to_string(index=False))

## Step 3: featureCounts でカウント行列生成

In [ ]:
%%time
bam_files = []
for _, row in metadata.iterrows():
    bam = os.path.join(MAPPED_DIR, f"{row['sample_id']}_Aligned.sortedByCoord.out.bam")
    assert os.path.exists(bam), f"BAMファイルが見つかりません: {bam}"
    bam_files.append(bam)

bam_str = " ".join(bam_files)
output_file = os.path.join(RESULTS_DIR, "featureCounts_output.txt")

cmd = (
    f"featureCounts -a {GTF_FILE} "
    f"-o {output_file} "
    f"-p --countReadPairs "
    f"-T {THREADS} "
    f"-s 0 "  # strandedness: 0=unstranded
    f"-B -C "  # only count proper pairs, exclude chimeric
    f"{bam_str}"
)

print("featureCounts 実行中...")
os.system(cmd)
print("featureCounts 完了")

## Step 4: カウント行列の整形

In [ ]:
# featureCountsの出力を読み込み、整形
fc_output = pd.read_csv(
    os.path.join(RESULTS_DIR, "featureCounts_output.txt"),
    sep="\t", comment="#"
)

# 遺伝子IDとカウントカラムのみ抽出
gene_col = "Geneid"
count_cols = [c for c in fc_output.columns if c.endswith(".bam")]

count_matrix = fc_output[[gene_col] + count_cols].copy()
count_matrix = count_matrix.set_index(gene_col)

# カラム名をサンプルIDに変換
new_names = {}
for col in count_cols:
    for _, row in metadata.iterrows():
        if row["sample_id"] in col:
            new_names[col] = row["sample_id"]
            break

count_matrix = count_matrix.rename(columns=new_names)

# サンプル順をメタデータに合わせる
count_matrix = count_matrix[metadata["sample_id"].tolist()]

# CSV出力
count_matrix.to_csv(os.path.join(RESULTS_DIR, "count_matrix.csv"))

print(f"カウント行列: {count_matrix.shape[0]} genes x {count_matrix.shape[1]} samples")
print(f"出力: {RESULTS_DIR}/count_matrix.csv")
print("\nサンプル別の総リードカウント:")
print(count_matrix.sum().to_string())

## 完了- `results/count_matrix.csv` が生成されました- 次のノートブック `03_DEG_Analysis.ipynb` に進んでください- Rカーネルに切り替えることを忘れないでください